# Caso C — AOV · Modelamiento desplegado

> **Qué hace este notebook.** Despliega los dos modelos de `run_case_c`: el **GLM inferencial** (drivers del ticket con significancia) y el **boosting predictivo** del gasto del cliente recurrente (features RFM + loyalty), con su HPO, comparación de modelos e interpretabilidad SHAP.

> **Cómo ejecutar.** `Restart & Run All`; determinista.

## 1. Datos: tickets enriquecidos

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_c, _ = masters.build_master_c(
        catalog.load('c_transacciones_resumen'), catalog.load('c_clientes_loyalty'),
        catalog.load('c_variables_exogenas'), catalog.load('c_promociones_activas'))
print('Master:', master_c.shape)

## 2. Modelo inferencial (GLM): ¿qué mueve el ticket?

Se winsorizan los outliers del ticket, se arma la matriz de diseño (one-hot de categóricas) y se ajusta un GLM gaussiano: coeficientes con error estándar, p-valor e IC95%.

In [ ]:
import pandas as pd
from tostao_ml.framework.io import set_global_seed
from tostao_ml.framework.features import Winsorizer
from tostao_ml.framework.models import GLMModel

set_global_seed(42)
TARGET = 'total_venta'
DRIVERS_NUM = ['total_articulos', 'edad', 'competitor_price_index', 'indice_trafico',
               'n_promos_activas', 'hora', 'dia_semana', 'antiguedad_cliente_dias']
DRIVERS_CAT = ['segmento', 'clima']

dfw = Winsorizer([TARGET], 0.01, 0.99).fit_transform(master_c)
num = [c for c in DRIVERS_NUM if c in dfw]
cat = [c for c in DRIVERS_CAT if c in dfw]
X = pd.get_dummies(dfw[num + cat], columns=cat, drop_first=True).astype(float)
yg = dfw[TARGET].astype(float)
glm = GLMModel(family='gaussian').fit(X, yg)
coefs = glm.coefficients_frame().drop(index='const', errors='ignore')
coefs = coefs.reindex(coefs['coef'].abs().sort_values(ascending=False).index)
coefs.round(4)

### Drivers significativos y figura de coeficientes

Un efecto es real si p<0.05 y su IC95% no cruza cero.

In [ ]:
import plotly.graph_objects as go

significativos = coefs[coefs['pvalue'] < 0.05]
print('Drivers significativos (p<0.05):', list(significativos.index))
top = coefs.reindex(coefs['coef'].abs().sort_values().index).tail(12)
fig = go.Figure(go.Bar(x=top['coef'], y=list(top.index), orientation='h'))
fig.add_vline(x=0, line_dash='dash')
fig.update_layout(title='Drivers del ticket (β, IC95%)', height=420)
fig.show()

## 3. Modelo predictivo del gasto: RFM + loyalty

Se calculan features RFM por cliente y se restringe a clientes **recurrentes** (≥2 visitas), cuyo gasto futuro es predecible por su perfil. Holdout 80/20.

In [ ]:
import numpy as np
from tostao_ml.framework.features import RFMTransformer

rfm = RFMTransformer('id_cliente', 'fecha', TARGET, score=False).fit_transform(master_c)
profile = master_c.groupby('id_cliente', observed=True).agg(
    edad=('edad', 'first'), segmento=('segmento', 'first'),
    antiguedad=('antiguedad_cliente_dias', 'max'), ticket_medio=('total_venta', 'mean'))
data = rfm.join(profile).dropna(subset=['ticket_medio'])
data = data[data['frequency'] >= 2]
yv = data['ticket_medio'].astype(float)
feat = pd.get_dummies(data[['recency', 'frequency', 'edad', 'antiguedad', 'segmento']],
                      columns=['segmento'], drop_first=True).astype(float)
rng = np.random.default_rng(42)
mask = rng.random(len(data)) < 0.8
print(f'clientes recurrentes: {len(data)}  train={int(mask.sum())}  test={int((~mask).sum())}')

## 4. HPO (Optuna, K-Fold) y ajuste del boosting

In [ ]:
from tostao_ml.framework.evaluation import metrics, kfold_splitter
from tostao_ml.framework.tuning import tune_model
from tostao_ml.framework.models import GBRRegressionModel

HPO_SPACE = {
    'learning_rate': {'type': 'float', 'low': 0.02, 'high': 0.3, 'log': True},
    'max_depth': {'type': 'int', 'low': 2, 'high': 8},
    'max_iter': {'type': 'int', 'low': 80, 'high': 350},
}
tuning = tune_model(
    'gbr', HPO_SPACE, feat[mask].reset_index(drop=True), yv[mask].reset_index(drop=True),
    scorer=lambda m, xv, yy: metrics.wape(yy, m.predict(xv)),
    splitter=kfold_splitter(n_splits=4, seed=42), direction='minimize',
    n_trials=20, sampler='tpe', pruner='none', seed=42)
hp = {k: tuning.best_params[k] for k in ('learning_rate', 'max_depth', 'max_iter')}
model = GBRRegressionModel(random_state=42, **hp).fit(feat[mask], yv[mask])
pred = model.predict(feat[~mask])
print('Mejor configuración:', hp)

## 5. Comparación de modelos y métricas del predictivo

In [ ]:
from tostao_ml.framework.models import RidgeRegressionModel, AveragingEnsemble
from tostao_ml.framework.evaluation import compare_models

candidatos = {
    'Ridge (lineal)': RidgeRegressionModel(random_state=42),
    'GBR': model,
    'Ensemble (Ridge+GBR)': AveragingEnsemble([
        ('ridge', RidgeRegressionModel(random_state=42)),
        ('gbr', GBRRegressionModel(random_state=42, **hp))]),
}
comparison = compare_models(candidatos, feat[mask], yv[mask], feat[~mask], yv[~mask], sort_by='wape')
display(comparison.round(4))
report = metrics.regression_report(yv[~mask].to_numpy(float), pred)
report['wape_baseline'] = metrics.wape(yv[~mask].to_numpy(float), np.full(len(pred), yv[mask].mean()))
pd.DataFrame([report]).T.rename(columns={0: 'valor'}).round(4)

## 6. Figuras de desempeño e interpretabilidad (SHAP)

In [ ]:
from tostao_ml.framework.evaluation import performance
from tostao_ml.framework.interpret import compute_shap, shap_summary_bar

y_test = yv[~mask].to_numpy(float)
performance.pred_vs_actual(y_test, pred).show()
performance.residuals_vs_pred(y_test, pred).show()
shap_res = compute_shap(model, feat[~mask], sample_size=120)
shap_summary_bar(shap_res).show()

## Conclusión

`total_articulos` es el driver dominante y altamente significativo; el clima lluvioso reduce el ticket con efecto pequeño. El modelo de gasto predice bien (R² fuerte) y prioriza clientes por valor esperado. Mismo modelamiento que persiste el pipeline en `data/06_models/modelo_caso_c.pkl`.